## 1. Environment Setup and Library Imports

In [ ]:
from __future__ import annotations
import re
import csv
import json
import math
import torch
import jieba
import joblib
import random
import hashlib
import warnings
import statistics
import numpy as np
import pandas as pd
import seaborn as sns
import torch.nn as nn
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import matplotlib.font_manager as fm
from pathlib import Path
from torch.optim import AdamW
from wordcloud import WordCloud
from sklearn.manifold import TSNE
from dataclasses import dataclass
from sklearn.cluster import KMeans
from datetime import date, datetime
from statistics import mean, median
from collections import Counter, defaultdict
from torch.utils.data import Dataset, DataLoader
from sklearn.feature_extraction.text import TfidfVectorizer
from transformers import AutoTokenizer, BertTokenizer, AutoModel, BertModel

try:
    import ijson
except Exception:
    ijson = None

warnings.filterwarnings("ignore", category=UserWarning)

## 2. Dataset Inventory and Format Sanity Check

In [ ]:
root = Path("data")
print("data_dir_abs", str(root.resolve()))
print("data_exists", root.exists())

rumor_dir = root / "CSDC-Rumor"
news_dir = root / "CSDC-News"
legal_file = root / "CSDC-Legal.json"

print("rumor_dir_exists", rumor_dir.exists())
print("news_dir_exists", news_dir.exists())
print("legal_file_exists", legal_file.exists())


def list_files(p: Path, pattern: str):
    if not p.exists():
        return []
    files = [x for x in p.glob(pattern) if x.is_file()]
    files.sort()
    return files


def total_bytes(files):
    s = 0
    for f in files:
        s += f.stat().st_size
    return s


rumor_weibo_dir = rumor_dir / "rumor_weibo"
rumor_fc_dir = rumor_dir / "rumor_forward_comment"
fact_file = rumor_dir / "fact.json"

news_data_dir = news_dir / "data"
news_comment_dir = news_dir / "comment"

rumor_weibo_files = list_files(rumor_weibo_dir, "*.json")
rumor_fc_files = list_files(rumor_fc_dir, "*.json")
news_data_files = list_files(news_data_dir, "*.txt")
news_comment_files = list_files(news_comment_dir, "*.txt")

print("fact_exists", fact_file.exists())
if fact_file.exists():
    print("fact_size_bytes", fact_file.stat().st_size)

print("rumor_weibo_files", len(rumor_weibo_files))
print("rumor_weibo_total_bytes", total_bytes(rumor_weibo_files))
print("rumor_weibo_samples", [f.name for f in rumor_weibo_files[:3]])

print("rumor_fc_files", len(rumor_fc_files))
print("rumor_fc_total_bytes", total_bytes(rumor_fc_files))
print("rumor_fc_samples", [f.name for f in rumor_fc_files[:3]])

print("news_data_files", len(news_data_files))
print("news_data_total_bytes", total_bytes(news_data_files))
print("news_data_samples", [f.name for f in news_data_files[:3]])

print("news_comment_files", len(news_comment_files))
print("news_comment_total_bytes", total_bytes(news_comment_files))
print("news_comment_samples", [f.name for f in news_comment_files[:3]])

if legal_file.exists():
    print("legal_size_bytes", legal_file.stat().st_size)

if fact_file.exists():
    with fact_file.open("r", encoding="utf-8", errors="ignore") as f:
        head_lines = []
        for _ in range(2):
            line = f.readline()
            if not line:
                break
            head_lines.append(line.strip())
    print("fact_head_lines", head_lines)

## 3. Single-Day CSDC-News Schema and Statistics Inspection

In [ ]:
root = Path("data") / "CSDC-News"
data_dir = root / "data"
comment_dir = root / "comment"

data_files = sorted([p for p in data_dir.glob("*.txt") if p.is_file()])
comment_files = sorted([p for p in comment_dir.glob("*.txt") if p.is_file()])

print("news_data_files", len(data_files))
print("news_comment_files", len(comment_files))

data_path = data_files[0] if data_files else None
comment_path = comment_files[0] if comment_files else None

print("sample_data_file", data_path.name if data_path else None)
print("sample_comment_file", comment_path.name if comment_path else None)


def iter_json_array(path: Path, limit=None):
    try:
        import ijson
    except Exception:
        ijson = None
    if ijson is not None:
        with path.open("rb") as f:
            for i, item in enumerate(ijson.items(f, "item")):
                yield item
                if limit is not None and i + 1 >= limit:
                    break
    else:
        with path.open("r", encoding="utf-8", errors="ignore") as f:
            arr = json.load(f)
        if limit is None:
            for item in arr:
                yield item
        else:
            for item in arr[:limit]:
                yield item


def safe_str(x):
    return "" if x is None else str(x)


covid_keywords = [
    "\u75ab\u60c5",
    "\u65b0\u51a0",
    "\u80ba\u708e",
    "\u51a0\u72b6\u75c5\u6bd2",
    "\u6b66\u6c49",
    "\u786e\u8bca",
    "\u53e3\u7f69",
    "\u9694\u79bb",
    "\u6838\u9178",
    "\u65b9\u8231",
]
covid_pattern = re.compile("|".join(re.escape(k) for k in covid_keywords))


def describe_lengths(values):
    values = [v for v in values if v is not None]
    if not values:
        return {"n": 0}
    values_sorted = sorted(values)
    n = len(values_sorted)

    def q(p):
        idx = int(round((n - 1) * p))
        return values_sorted[idx]

    return {
        "n": n,
        "mean": float(statistics.mean(values_sorted)),
        "median": float(statistics.median(values_sorted)),
        "p90": float(q(0.90)),
        "p95": float(q(0.95)),
        "max": float(values_sorted[-1]),
    }


if data_path is not None:
    items = list(iter_json_array(data_path, limit=None))
    print("news_items", len(items))

    keys = Counter()
    meta_keys = Counter()
    missing = Counter()
    title_lens = []
    content_lens = []
    covid_hits = 0
    type_counter = Counter()

    for it in items:
        if isinstance(it, dict):
            for k in it.keys():
                keys[k] += 1
            t = safe_str(it.get("title"))
            u = safe_str(it.get("url"))
            tm = safe_str(it.get("time"))
            meta = it.get("meta") if isinstance(it.get("meta"), dict) else {}
            for mk in meta.keys():
                meta_keys[mk] += 1

            if not t:
                missing["title"] += 1
            if not u:
                missing["url"] += 1
            if not tm:
                missing["time"] += 1
            c = safe_str(meta.get("content"))
            d = safe_str(meta.get("description"))
            if not c:
                missing["meta.content"] += 1
            if not d:
                missing["meta.description"] += 1

            title_lens.append(len(t))
            content_lens.append(len(c))

            text_for_match = (t + " " + c).strip()
            if text_for_match and covid_pattern.search(text_for_match) is not None:
                covid_hits += 1

            type_counter[safe_str(meta.get("type"))] += 1

    print("news_top_level_keys", dict(keys.most_common(20)))
    print("news_meta_keys", dict(meta_keys.most_common(20)))
    print("news_missing_counts", dict(missing))
    print("news_title_len_stats", describe_lengths(title_lens))
    print("news_content_len_stats", describe_lengths(content_lens))
    print("news_covid_hit_ratio", covid_hits / max(1, len(items)))
    print("news_top_types", dict(type_counter.most_common(10)))

if comment_path is not None:
    items = list(iter_json_array(comment_path, limit=None))
    print("comment_news_items", len(items))

    comment_counts = []
    area_counter = Counter()
    news_with_comments = 0

    for it in items:
        if not isinstance(it, dict):
            continue
        comments = it.get("comment")
        if not isinstance(comments, list):
            continue
        if len(comments) > 0:
            news_with_comments += 1
        comment_counts.append(len(comments))
        for c in comments:
            if isinstance(c, dict):
                area_counter[safe_str(c.get("area"))] += 1

    print("comment_news_with_comments", news_with_comments)
    print("comment_count_stats", describe_lengths(comment_counts))
    print("comment_top_areas", dict(area_counter.most_common(15)))

## 4. Daily COVID News Trend

In [ ]:
step_dir = Path("outputs") / "step_03_news_covid_trend"
step_dir.mkdir(parents=True, exist_ok=True)

root = Path("data") / "CSDC-News" / "data"
files = sorted([p for p in root.glob("*.txt") if p.is_file()])

covid_keywords = [
    "\u75ab\u60c5",
    "\u65b0\u51a0",
    "\u80ba\u708e",
    "\u51a0\u72b6\u75c5\u6bd2",
    "\u65b0\u578b\u51a0\u72b6\u75c5\u6bd2",
    "\u786e\u8bca",
    "\u9694\u79bb",
    "\u53e3\u7f69",
    "\u6838\u9178",
    "\u65b9\u8231",
    "\u6b66\u6c49",
    "\u75ab\u82d7",
    "\u9632\u63a7",
]
covid_pattern = re.compile("|".join(re.escape(k) for k in covid_keywords))


def iter_items(path: Path):
    try:
        import ijson
    except Exception:
        ijson = None

    if ijson is not None:
        with path.open("rb") as f:
            for item in ijson.items(f, "item"):
                yield item
    else:
        with path.open("r", encoding="utf-8", errors="ignore") as f:
            arr = json.load(f)
        for item in arr:
            yield item


def safe_str(x):
    return "" if x is None else str(x)


rows = []
type_total = Counter()

for p in files:
    mmdd = p.stem
    date = pd.to_datetime("2020-" + mmdd, errors="coerce")
    total_count = 0
    covid_count = 0
    missing_content_count = 0
    type_counter = Counter()

    for it in iter_items(p):
        if not isinstance(it, dict):
            continue
        total_count += 1
        meta = it.get("meta") if isinstance(it.get("meta"), dict) else {}
        title = safe_str(it.get("title"))
        content = safe_str(meta.get("content"))
        if not content:
            missing_content_count += 1
        text = (title + " " + content).strip()
        if text and covid_pattern.search(text) is not None:
            covid_count += 1
        t = safe_str(meta.get("type"))
        type_counter[t] += 1
        type_total[t] += 1

    rows.append(
        {
            "date": date,
            "file": p.name,
            "total_count": int(total_count),
            "covid_count": int(covid_count),
            "covid_ratio": float(covid_count / total_count) if total_count else 0.0,
            "missing_content_count": int(missing_content_count),
            "top_type": type_counter.most_common(1)[0][0] if type_counter else "",
        }
    )

df = pd.DataFrame(rows).dropna(subset=["date"]).sort_values("date")
df["covid_count_ma7"] = df["covid_count"].rolling(7, min_periods=1).mean()
df["covid_ratio_ma7"] = df["covid_ratio"].rolling(7, min_periods=1).mean()

csv_path = step_dir / "news_daily_covid_trend.csv"
df.to_csv(csv_path, index=False, encoding="utf-8-sig")

total_news = int(df["total_count"].sum())
total_covid = int(df["covid_count"].sum())
overall_ratio = float(total_covid / max(1, total_news))

peak_count_row = df.loc[
    df["covid_count"].idxmax(),
    ["date", "file", "total_count", "covid_count", "covid_ratio"],
]
peak_ratio_row = df.loc[
    df["covid_ratio"].idxmax(),
    ["date", "file", "total_count", "covid_count", "covid_ratio"],
]

thresholds = [0.01, 0.03, 0.05, 0.10, 0.30, 0.50, 0.70]
threshold_hits = {}
for th in thresholds:
    hit = df[df["covid_ratio"] >= th]
    threshold_hits[str(th)] = str(hit["date"].iloc[0]) if len(hit) else ""

summary_lines = []
summary_lines.append(f"news_data_files={len(files)}")
summary_lines.append(f"ijson_available={ijson is not None}")
summary_lines.append(f"date_min={df['date'].min()}")
summary_lines.append(f"date_max={df['date'].max()}")
summary_lines.append(f"total_news_items={total_news}")
summary_lines.append(f"total_covid_items={total_covid}")
summary_lines.append(f"overall_covid_ratio={overall_ratio}")
summary_lines.append(f"peak_by_covid_count={peak_count_row.to_dict()}")
summary_lines.append(f"peak_by_covid_ratio={peak_ratio_row.to_dict()}")
summary_lines.append(f"type_total_top10={dict(type_total.most_common(10))}")
summary_lines.append(f"first_date_ratio_crossing={threshold_hits}")

summary_path = step_dir / "summary.txt"
summary_path.write_text("\n".join(summary_lines), encoding="utf-8")

try:
    plt.style.use("seaborn-v0_8-whitegrid")
except Exception:
    pass

locator = mdates.AutoDateLocator(minticks=6, maxticks=10)
formatter = mdates.ConciseDateFormatter(locator)

fig, ax = plt.subplots(figsize=(12, 5), constrained_layout=True)
ax.plot(df["date"], df["covid_count"], linewidth=1.2, alpha=0.55, label="daily")
ax.plot(df["date"], df["covid_count_ma7"], linewidth=2.0, label="7d_mean")
ax.set_title("Daily COVID-related News Count")
ax.set_xlabel("date")
ax.set_ylabel("count")
ax.xaxis.set_major_locator(locator)
ax.xaxis.set_major_formatter(formatter)
ax.legend(frameon=False, loc="upper right")
ax.grid(True, alpha=0.25)
count_png = step_dir / "news_daily_covid_count.png"
fig.savefig(count_png, dpi=300)
plt.close(fig)

fig, ax = plt.subplots(figsize=(12, 5), constrained_layout=True)
ax.plot(df["date"], df["covid_ratio"], linewidth=1.2, alpha=0.55, label="daily")
ax.plot(df["date"], df["covid_ratio_ma7"], linewidth=2.0, label="7d_mean")
ax.set_title("Daily COVID-related News Ratio")
ax.set_xlabel("date")
ax.set_ylabel("ratio")
ax.set_ylim(0.0, 1.0)
ax.xaxis.set_major_locator(locator)
ax.xaxis.set_major_formatter(formatter)
ax.legend(frameon=False, loc="upper right")
ax.grid(True, alpha=0.25)
ratio_png = step_dir / "news_daily_covid_ratio.png"
fig.savefig(ratio_png, dpi=300)
plt.close(fig)

print("saved_step_dir", str(step_dir.resolve()))
print("saved_csv", str(csv_path.resolve()))
print("saved_summary", str(summary_path.resolve()))
print("saved_png", str(count_png.resolve()))
print("saved_png", str(ratio_png.resolve()))

## 5. COVID Keyword Corpus and Word Cloud Generation

In [ ]:
step_dir = Path("outputs") / "step_04_covid_keywords"
step_dir.mkdir(parents=True, exist_ok=True)

data_dir = Path("data") / "CSDC-News" / "data"
files = sorted([p for p in data_dir.glob("*.txt") if p.is_file()])

covid_keywords = [
    "\u75ab\u60c5",
    "\u65b0\u51a0",
    "\u80ba\u708e",
    "\u51a0\u72b6\u75c5\u6bd2",
    "\u786e\u8bca",
    "\u9694\u79bb",
    "\u53e3\u7f69",
    "\u75ab\u82d7",
    "\u9632\u63a7",
]
covid_pattern = re.compile("|".join(re.escape(k) for k in covid_keywords))


def iter_items(path: Path):
    if ijson is not None:
        with path.open("rb") as f:
            for item in ijson.items(f, "item"):
                yield item
    else:
        with path.open("r", encoding="utf-8", errors="ignore") as f:
            arr = json.load(f)
        for item in arr:
            yield item


def safe_list(x):
    return x if isinstance(x, list) else []


counter = Counter()

for p in files:
    for it in iter_items(p):
        if not isinstance(it, dict):
            continue
        meta = it.get("meta") if isinstance(it.get("meta"), dict) else {}
        title = str(it.get("title", ""))
        content = str(meta.get("content", ""))
        text = (title + " " + content).strip()
        if not text:
            continue
        if covid_pattern.search(text) is None:
            continue
        for kw in safe_list(meta.get("keyword")):
            kw = str(kw).strip()
            if kw:
                counter[kw] += 1

df_kw = pd.DataFrame(counter.most_common(), columns=["keyword", "count"])
csv_path = step_dir / "covid_keyword_frequency.csv"
df_kw.to_csv(csv_path, index=False, encoding="utf-8-sig")

font_path = "./SimHei.ttf"
wc = WordCloud(
    font_path=font_path,
    width=1600,
    height=1000,
    background_color="white",
    max_words=200,
    colormap="tab10",
)

wc.generate_from_frequencies(dict(counter))

fig, ax = plt.subplots(figsize=(10, 6), constrained_layout=True)
ax.imshow(wc, interpolation="bilinear")
ax.axis("off")
out_png = step_dir / "covid_keyword_wordcloud.png"
fig.savefig(out_png, dpi=300)
plt.close(fig)

print("saved_step_dir", str(step_dir.resolve()))
print("saved_csv", str(csv_path.resolve()))
print("saved_png", str(out_png.resolve()))
print("top_20_keywords", df_kw.head(20).to_dict(orient="records"))

## 6. Weakly Supervised BERT Sentiment Comparison

In [ ]:
STEP_DIR = Path("outputs") / "step_05_sentiment_analysis"
STEP_DIR.mkdir(parents=True, exist_ok=True)

BERT_PATH = Path("./chinese-bert-wwm-ext")
DEVICE = torch.device("cuda")

MAX_LEN = 256
TRAIN_BATCH_SIZE = 128
INF_BATCH_SIZE = 1024
EPOCHS = 3
LEARNING_RATE = 1e-5
SEED = 42
SAMPLE_LIMIT = 1e20
NUM_WORKERS = 16

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.benchmark = True

print(f"Device: {DEVICE} ({torch.cuda.get_device_name(0)})")
print(
    f"Config: Batch={TRAIN_BATCH_SIZE}/{INF_BATCH_SIZE}, Len={MAX_LEN}, Workers={NUM_WORKERS}, AMP=True"
)


def clean_text(text):
    if not text:
        return ""
    text = str(text)
    text = re.sub(r"http\S+", "", text)
    text = re.sub(r"@\S+", "", text)
    text = re.sub(r"\s+", " ", text)
    return text.strip()


def get_sentiment_label(text):
    pos_seeds = [
        "加油",
        "致敬",
        "感动",
        "支持",
        "正能量",
        "谢谢",
        "辛苦",
        "赞",
        "好样",
        "英雄",
        "早日康复",
        "必胜",
        "良心",
        "希望",
    ]
    neg_seeds = [
        "造谣",
        "假新闻",
        "恶心",
        "垃圾",
        "严惩",
        "愤怒",
        "寒心",
        "无耻",
        "忽悠",
        "瞎编",
        "谣言",
        "甚至",
        "失望",
        "不管",
    ]

    if any(s in text for s in pos_seeds):
        return 1
    if any(s in text for s in neg_seeds):
        return 0
    return -1


def load_news_comments(limit):
    comments = []
    root = Path("data") / "CSDC-News" / "comment"
    files = sorted([p for p in root.glob("*.txt") if p.is_file()])

    count = 0
    for p in files:
        if count >= limit:
            break
        try:
            with p.open("r", encoding="utf-8", errors="ignore") as f:
                data = json.load(f)
                for item in data:
                    item_comments = item.get("comment", [])
                    if isinstance(item_comments, list):
                        for c in item_comments:
                            if isinstance(c, dict):
                                txt = clean_text(c.get("content"))
                                if len(txt) > 2:
                                    comments.append({"text": txt, "source": "news"})
                                    count += 1
                                    if count >= limit:
                                        break
                    if count >= limit:
                        break
        except Exception:
            continue
    return comments


def load_rumor_comments(limit):
    comments = []
    root = Path("data") / "CSDC-Rumor" / "rumor_forward_comment"
    files = sorted([p for p in root.glob("*.json") if p.is_file()])

    count = 0
    for p in files:
        if count >= limit:
            break
        try:
            with p.open("r", encoding="utf-8", errors="ignore") as f:
                data = json.load(f)
                if isinstance(data, list):
                    for item in data:
                        txt = clean_text(item.get("text"))
                        if len(txt) > 2:
                            comments.append({"text": txt, "source": "rumor"})
                            count += 1
        except Exception:
            continue
    return comments


print("Loading large-scale data...")
news_data = load_news_comments(limit=SAMPLE_LIMIT)
rumor_data = load_rumor_comments(limit=SAMPLE_LIMIT)
all_data = news_data + rumor_data
print(f"Loaded {len(news_data)} news comments and {len(rumor_data)} rumor comments.")

train_texts = []
train_labels = []

for item in all_data:
    label = get_sentiment_label(item["text"])
    if label != -1:
        train_texts.append(item["text"])
        train_labels.append(label)

print(f"Training set size (Weakly Labeled): {len(train_texts)}")
print(f"Training Class Distribution: {Counter(train_labels)}")


class SentimentDataset(Dataset):
    def __init__(self, texts, labels=None, tokenizer=None, max_len=128):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = str(self.texts[idx])
        encoding = self.tokenizer(
            text,
            add_special_tokens=True,
            max_length=self.max_len,
            padding="max_length",
            truncation=True,
            return_attention_mask=True,
            return_tensors="pt",
        )

        item = {
            "input_ids": encoding["input_ids"].flatten(),
            "attention_mask": encoding["attention_mask"].flatten(),
        }

        if self.labels is not None:
            item["label"] = torch.tensor(self.labels[idx], dtype=torch.long)

        return item


class BertClassifier(nn.Module):
    def __init__(self, bert_path):
        super(BertClassifier, self).__init__()
        self.bert = BertModel.from_pretrained(bert_path)
        self.drop = nn.Dropout(p=0.3)
        self.out = nn.Linear(self.bert.config.hidden_size, 2)

    def forward(self, input_ids, attention_mask):
        outputs = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        pooled_output = outputs.pooler_output
        output = self.drop(pooled_output)
        return self.out(output)


tokenizer = BertTokenizer.from_pretrained(BERT_PATH)
train_dataset = SentimentDataset(train_texts, train_labels, tokenizer, MAX_LEN)

train_loader = DataLoader(
    train_dataset,
    batch_size=TRAIN_BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=True,
)

model = BertClassifier(BERT_PATH).to(DEVICE)
optimizer = AdamW(model.parameters(), lr=LEARNING_RATE)
loss_fn = nn.CrossEntropyLoss().to(DEVICE)
scaler = torch.amp.GradScaler("cuda")

print("Starting Mixed-Precision Fine-tuning...")
model.train()
for epoch in range(EPOCHS):
    total_loss = 0
    correct = 0
    total = 0
    for batch in train_loader:
        input_ids = batch["input_ids"].to(DEVICE)
        attention_mask = batch["attention_mask"].to(DEVICE)
        labels = batch["label"].to(DEVICE)

        optimizer.zero_grad()

        with torch.amp.autocast("cuda"):
            outputs = model(input_ids, attention_mask)
            loss = loss_fn(outputs, labels)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        total_loss += loss.item()
        _, predicted = torch.max(outputs, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

    print(
        f"Epoch {epoch + 1}/{EPOCHS} | Loss: {total_loss / len(train_loader):.4f} | Acc: {correct / total:.4f}"
    )

torch.save(model.state_dict(), STEP_DIR / "sentiment_model.pt")

print("Starting High-Speed Inference...")
inference_dataset = SentimentDataset(
    [x["text"] for x in all_data], None, tokenizer, MAX_LEN
)
inference_loader = DataLoader(
    inference_dataset,
    batch_size=INF_BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=True,
)

model.eval()
probabilities = []
with torch.no_grad():
    for batch in inference_loader:
        input_ids = batch["input_ids"].to(DEVICE)
        attention_mask = batch["attention_mask"].to(DEVICE)

        with torch.amp.autocast("cuda"):
            outputs = model(input_ids, attention_mask)
            probs = torch.softmax(outputs, dim=1)

        probabilities.extend(probs[:, 1].cpu().float().numpy())

df_res = pd.DataFrame(all_data)
df_res["pos_prob"] = probabilities

print("Generating visualizations...")
stats = df_res.groupby("source")["pos_prob"].describe()
print(stats)
stats.to_csv(STEP_DIR / "sentiment_comparison.csv")

plt.style.use("seaborn-v0_8-whitegrid")
fig, ax = plt.subplots(figsize=(12, 7), constrained_layout=True)
sns.kdeplot(
    data=df_res[df_res["source"] == "news"],
    x="pos_prob",
    fill=True,
    label="News Comments",
    color="#1f77b4",
    alpha=0.3,
    linewidth=2.5,
    ax=ax,
)
sns.kdeplot(
    data=df_res[df_res["source"] == "rumor"],
    x="pos_prob",
    fill=True,
    label="Rumor Comments",
    color="#d62728",
    alpha=0.3,
    linewidth=2.5,
    ax=ax,
)

ax.set_title("Sentiment Polarity Distribution", fontsize=16, pad=15)
ax.set_xlabel("Positive Sentiment Probability", fontsize=14)
ax.set_ylabel("Density", fontsize=14)
ax.set_xlim(0, 1)
ax.legend(fontsize=13, frameon=False)
ax.grid(True, alpha=0.2)
plot_path = STEP_DIR / "sentiment_distribution.png"
fig.savefig(plot_path, dpi=300)
plt.close(fig)

print("saved_step_dir", str(STEP_DIR.resolve()))

## 7. Social Mood Timeline and Attention Comparison

In [ ]:
STEP_DIR = Path("outputs") / "step_06_temporal_mood"
STEP_DIR.mkdir(parents=True, exist_ok=True)

MODEL_PATH = Path("outputs") / "step_05_sentiment_analysis" / "sentiment_model.pt"
TREND_CSV = Path("outputs") / "step_03_news_covid_trend" / "news_daily_covid_trend.csv"
BERT_PATH = Path("./chinese-bert-wwm-ext")
DEVICE = torch.device("cuda")

MAX_LEN = 256
BATCH_SIZE = 2048
NUM_WORKERS = 16

print(f"Device: {DEVICE}")
print(f"Loading Trend Data from: {TREND_CSV}")


def load_news_with_date():
    items = []
    root = Path("data") / "CSDC-News" / "comment"
    files = sorted([p for p in root.glob("*.txt") if p.is_file()])

    for p in files:
        date_str = "2020-" + p.stem
        try:
            with p.open("r", encoding="utf-8", errors="ignore") as f:
                data = json.load(f)
                for item in data:
                    item_comments = item.get("comment", [])
                    if isinstance(item_comments, list):
                        for c in item_comments:
                            if isinstance(c, dict):
                                txt = clean_text(c.get("content"))
                                if len(txt) > 2:
                                    items.append(
                                        {
                                            "text": txt,
                                            "date": date_str,
                                            "source": "news",
                                        }
                                    )
        except Exception:
            continue
    return items


def load_rumor_with_date():
    items = []
    root = Path("data") / "CSDC-Rumor" / "rumor_forward_comment"
    files = sorted([p for p in root.glob("*.json") if p.is_file()])

    for p in files:
        try:
            with p.open("r", encoding="utf-8", errors="ignore") as f:
                data = json.load(f)
                if isinstance(data, list):
                    for item in data:
                        txt = clean_text(item.get("text"))
                        d_str = str(item.get("date", "")).split()[0]
                        if len(txt) > 2 and len(d_str) >= 10:
                            items.append(
                                {"text": txt, "date": d_str, "source": "rumor"}
                            )
        except Exception:
            continue
    return items


print("Reloading data with timestamps...")
news_items = load_news_with_date()
rumor_items = load_rumor_with_date()
all_items = news_items + rumor_items
print(f"Total timestamped items: {len(all_items)}")


class InferenceDataset(Dataset):
    def __init__(self, texts, tokenizer, max_len=128):
        self.texts = texts
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = str(self.texts[idx])
        encoding = self.tokenizer(
            text,
            add_special_tokens=True,
            max_length=self.max_len,
            padding="max_length",
            truncation=True,
            return_attention_mask=True,
            return_tensors="pt",
        )
        return {
            "input_ids": encoding["input_ids"].flatten(),
            "attention_mask": encoding["attention_mask"].flatten(),
        }


class BertClassifier(nn.Module):
    def __init__(self, bert_path):
        super(BertClassifier, self).__init__()
        self.bert = BertModel.from_pretrained(bert_path)
        self.drop = nn.Dropout(p=0.3)
        self.out = nn.Linear(self.bert.config.hidden_size, 2)

    def forward(self, input_ids, attention_mask):
        outputs = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        output = self.drop(outputs.pooler_output)
        return self.out(output)


print("Loading fine-tuned model...")
tokenizer = BertTokenizer.from_pretrained(BERT_PATH)
model = BertClassifier(BERT_PATH)
model.load_state_dict(torch.load(MODEL_PATH, map_location=DEVICE))
model.to(DEVICE)
model.eval()

dataset = InferenceDataset([x["text"] for x in all_items], tokenizer, MAX_LEN)
loader = DataLoader(
    dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=True,
)

print("Running Time-Series Inference...")
probs = []
with torch.no_grad():
    for batch in loader:
        input_ids = batch["input_ids"].to(DEVICE)
        attention_mask = batch["attention_mask"].to(DEVICE)
        with torch.amp.autocast("cuda"):
            outputs = model(input_ids, attention_mask)
            p = torch.softmax(outputs, dim=1)[:, 1]
        probs.extend(p.cpu().float().numpy())

df = pd.DataFrame(all_items)
df["sentiment"] = probs
df["date"] = pd.to_datetime(df["date"], errors="coerce")
df = df.dropna(subset=["date"])

daily_sentiment = df.groupby("date")["sentiment"].mean().reset_index()
daily_sentiment.columns = ["date", "sentiment_score"]

if TREND_CSV.exists():
    df_trend = pd.read_csv(TREND_CSV)
    df_trend["date"] = pd.to_datetime(df_trend["date"])
    merged = pd.merge(df_trend, daily_sentiment, on="date", how="outer").sort_values(
        "date"
    )
else:
    print("Warning: Step 3 Trend CSV not found, plotting sentiment only.")
    merged = daily_sentiment.sort_values("date")

merged = merged[(merged["date"] >= "2020-01-01") & (merged["date"] <= "2020-06-30")]
merged["sentiment_ma7"] = merged["sentiment_score"].rolling(7).mean()
merged["covid_count_ma7"] = (
    merged["covid_count"].rolling(7).mean() if "covid_count" in merged.columns else 0
)

merged.to_csv(STEP_DIR / "daily_sentiment.csv", index=False)
print("Saved daily sentiment data.")

plt.style.use("seaborn-v0_8-whitegrid")
fig, ax1 = plt.subplots(figsize=(14, 7), constrained_layout=True)

color_sent = "#d62728"
color_news = "#1f77b4"

ax1.set_xlabel("Date", fontsize=14)
ax1.set_ylabel("Social Mood Index (Higher is Better)", color=color_sent, fontsize=14)
l1 = ax1.plot(
    merged["date"],
    merged["sentiment_ma7"],
    color=color_sent,
    linewidth=3,
    label="Mood Index (7d MA)",
)
ax1.tick_params(axis="y", labelcolor=color_sent)
ax1.set_ylim(0.1, 0.6)

ax2 = ax1.twinx()
ax2.set_ylabel("COVID News Attention (Volume)", color=color_news, fontsize=14)
l2 = ax2.fill_between(
    merged["date"],
    0,
    merged["covid_count_ma7"],
    color=color_news,
    alpha=0.15,
    label="News Attention",
)
ax2.tick_params(axis="y", labelcolor=color_news)

ax1.set_title(
    "Temporal Evolution: Social Mood vs. Pandemic Attention (Jan - Jun 2020)",
    fontsize=18,
    pad=20,
)
ax1.grid(True, alpha=0.3)

lines = l1 + [l2]
labels = [l.get_label() for l in lines]
ax1.legend(lines, labels, loc="upper center", fontsize=12, frameon=True, ncol=2)

date_fmt = mdates.DateFormatter("%b-%d")
ax1.xaxis.set_major_formatter(date_fmt)
ax1.xaxis.set_major_locator(mdates.DayLocator(interval=14))

out_png = STEP_DIR / "mood_vs_attention.png"
fig.savefig(out_png, dpi=300)
plt.close(fig)

print("saved_step_dir", str(STEP_DIR.resolve()))
print("saved_png", str(out_png.resolve()))

data_dir = Path("data") / "CSDC-News" / "data"
comment_dir = Path("data") / "CSDC-News" / "comment"


def get_date_range(folder_path):
    if not folder_path.exists():
        return "Not Found", 0, "N/A", "N/A"

    dates = []
    files = sorted([p.name for p in folder_path.glob("*.txt")])

    for fname in files:
        try:
            date_str = "2020-" + fname.replace(".txt", "")
            dt = datetime.strptime(date_str, "%Y-%m-%d").date()
            dates.append(dt)
        except ValueError:
            continue

    if not dates:
        return "Empty", 0, "N/A", "N/A"

    dates.sort()
    return "OK", len(dates), dates[0], dates[-1]


d_status, d_count, d_start, d_end = get_date_range(data_dir)
c_status, c_count, c_start, c_end = get_date_range(comment_dir)

print("-" * 50)
print(
    f"{'Dataset':<15} | {'Status':<10} | {'Files':<5} | {'Start Date':<12} | {'End Date':<12}"
)
print("-" * 50)
print(f"{'News Data':<15} | {d_status:<10} | {d_count:<5} | {d_start}   | {d_end}")
print(f"{'News Comments':<15} | {c_status:<10} | {c_count:<5} | {c_start}   | {c_end}")
print("-" * 50)

if c_end != "N/A" and d_end != "N/A":
    if c_end < d_end:
        print("\n[VERIFIED] Comment data ends earlier than News data.")
        print(f"Gap: Comments miss data from {c_end} to {d_end}.")
    else:
        print("\n[RESULT] Date ranges match.")

## 8. Law and Rumor Semantic Landscape

In [ ]:
STEP_DIR = Path("outputs") / "step_07_legal_rumor"
STEP_DIR.mkdir(parents=True, exist_ok=True)

BERT_PATH = Path("./chinese-bert-wwm-ext")
LEGAL_PATH = Path("data") / "CSDC-Legal.json"
RUMOR_FACT_PATH = Path("data") / "CSDC-Rumor" / "fact.json"
FONT_PATH = Path("SimHei.ttf")
DEVICE = torch.device("cuda")

BATCH_SIZE = 128
MAX_LEN = 512
N_CLUSTERS_LEGAL = 10
SEED = 42

np.random.seed(SEED)
torch.manual_seed(SEED)


if FONT_PATH.exists():
    my_font = fm.FontProperties(fname=str(FONT_PATH))
    print(f"Loaded font from {FONT_PATH}")
else:
    print("Warning: SimHei.ttf not found! Chinese labels may fail.")
    my_font = None

print(f"Device: {DEVICE}")


def load_legal_data():
    with open(LEGAL_PATH, "r", encoding="utf-8") as f:
        data = json.load(f)
    texts = []
    ids = []
    for item in data:
        t = str(item.get("title", "")) + " " + str(item.get("content", ""))[:500]
        texts.append(t.strip())
        ids.append(item.get("document_id", "unknown"))
    return texts, ids


def load_rumor_facts():
    texts = []
    labels = []
    with open(RUMOR_FACT_PATH, "r", encoding="utf-8") as f:
        for line in f:
            if not line.strip():
                continue
            try:
                obj = json.loads(line)
                t = str(obj.get("title", "")) + " " + str(obj.get("rumor", ""))
                texts.append(t.strip())
                labels.append(obj.get("explain", "Unknown"))
            except Exception:
                continue
    return texts, labels


print("Loading Data...")
legal_texts, legal_ids = load_legal_data()
rumor_texts, rumor_labels = load_rumor_facts()


class TextDataset(Dataset):
    def __init__(self, texts, tokenizer, max_len):
        self.texts = texts
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = str(self.texts[idx])
        encoding = self.tokenizer(
            text,
            add_special_tokens=True,
            max_length=self.max_len,
            padding="max_length",
            truncation=True,
            return_attention_mask=True,
            return_tensors="pt",
        )
        return {
            "input_ids": encoding["input_ids"].flatten(),
            "attention_mask": encoding["attention_mask"].flatten(),
        }


class BertEmbedder(nn.Module):
    def __init__(self, bert_path):
        super().__init__()
        self.bert = BertModel.from_pretrained(bert_path)

    def forward(self, input_ids, attention_mask):
        outputs = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        return outputs.pooler_output


def get_embeddings(texts):
    tokenizer = BertTokenizer.from_pretrained(BERT_PATH)
    dataset = TextDataset(texts, tokenizer, MAX_LEN)
    loader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=8)
    model = BertEmbedder(BERT_PATH).to(DEVICE)
    model.eval()
    embeds = []
    with torch.no_grad():
        for batch in loader:
            input_ids = batch["input_ids"].to(DEVICE)
            mask = batch["attention_mask"].to(DEVICE)
            with torch.amp.autocast("cuda"):
                out = model(input_ids, mask)
            embeds.append(out.cpu().numpy())
    return np.vstack(embeds)


print("Vectorizing...")
legal_vecs = get_embeddings(legal_texts)
rumor_vecs = get_embeddings(rumor_texts)

print("Clustering Legal Themes...")
kmeans = KMeans(n_clusters=N_CLUSTERS_LEGAL, random_state=SEED, n_init=10)
legal_clusters = kmeans.fit_predict(legal_vecs)


def extract_cluster_keywords(texts, clusters, n_clusters):
    df = pd.DataFrame({"text": texts, "cluster": clusters})
    cluster_keywords = {}

    stopwords = [
        "被告人",
        "本院",
        "判决",
        "认为",
        "依法",
        "公诉",
        "机关",
        "下列",
        "某某",
        "或者",
        "期间",
        "一案",
        "年月日",
        "受理",
        "发生",
        "汉族",
        "出生",
        "住",
        "委托",
        "代理人",
        "原告",
        "被告",
        "上诉人",
        "被上诉人",
        "终结",
        "审理",
        "判决书",
        "裁定书",
        "诉讼",
        "代理",
        "法定",
        "代表",
        "事实",
        "证据",
        "指控",
        "辩护",
        "确认",
        "驳回",
        "文化",
        "程度",
        "户籍",
        "刑事",
        "民事",
        "身份证",
        "号码",
    ]

    for i in range(n_clusters):
        cluster_txts = df[df["cluster"] == i]["text"].tolist()
        if not cluster_txts:
            continue

        tfidf = TfidfVectorizer(
            max_features=10, stop_words=stopwords, ngram_range=(1, 2)
        )
        try:
            tfidf.fit(cluster_txts)
            scores = zip(
                tfidf.get_feature_names_out(),
                tfidf.transform(cluster_txts).sum(axis=0).tolist()[0],
            )
            sorted_scores = sorted(scores, key=lambda x: x[1], reverse=True)
            cluster_keywords[i] = " | ".join([w for w, s in sorted_scores[:3]])
        except Exception:
            cluster_keywords[i] = "Misc"
    return cluster_keywords


cluster_names = extract_cluster_keywords(legal_texts, legal_clusters, N_CLUSTERS_LEGAL)
print("\n=== Refined Legal Clusters ===")
for cid, keywords in cluster_names.items():
    print(f"Cluster {cid}: {keywords}")

print("Projecting Semantic Landscape...")
all_vecs = np.vstack([legal_vecs, rumor_vecs])
tsne = TSNE(
    n_components=2, random_state=SEED, perplexity=30, init="pca", learning_rate="auto"
)
all_2d = tsne.fit_transform(all_vecs)
legal_2d = all_2d[: len(legal_vecs)]
rumor_2d = all_2d[len(legal_vecs) :]

plt.style.use("seaborn-v0_8-whitegrid")
fig, ax = plt.subplots(figsize=(16, 10), constrained_layout=True)

colors = sns.color_palette("tab10", N_CLUSTERS_LEGAL)
for i in range(N_CLUSTERS_LEGAL):
    mask = legal_clusters == i
    ax.scatter(
        legal_2d[mask, 0],
        legal_2d[mask, 1],
        c=[colors[i]],
        label=f"C{i}: {cluster_names[i]}",
        alpha=0.6,
        s=50,
    )

ax.scatter(
    rumor_2d[:, 0],
    rumor_2d[:, 1],
    c="red",
    label="Rumors / Misinfo",
    alpha=0.5,
    s=30,
    marker="x",
)

ax.set_title(
    "Semantic Landscape: Pandemic Law & Disorder (BERT + t-SNE)",
    fontsize=20,
    fontproperties=my_font,
)
ax.legend(
    loc="upper right", frameon=True, fontsize=11, prop=my_font, bbox_to_anchor=(1.25, 1)
)

out_png = STEP_DIR / "semantic_landscape.png"
fig.savefig(out_png, dpi=300, bbox_inches="tight")
plt.close(fig)

df_legal = pd.DataFrame(
    {
        "id": legal_ids,
        "cluster": legal_clusters,
        "theme": [cluster_names[c] for c in legal_clusters],
    }
)
df_legal.to_csv(STEP_DIR / "legal_clusters.csv", index=False)
print(f"Done. Check {out_png}")